# 05. 연결 확인과 에러 처리

API 요청이 실패하는 경우를 하나씩 **일부러 재현**해 보고, 어떤 예외가 발생하는지 확인한다.

## 먼저 알아둘 것

- **이 노트북은 대부분 무료다.** 모델을 실행하지 않는 요청(`models.list`, `count_tokens`)으로 에러를 재현한다. 요금이 발생하는 셀은 따로 표시했다.
- 실패는 크게 두 단계로 나뉜다.
  1. **서버에 닿지 못함**: 네트워크 끊김, 시간 초과 → `APIConnectionError`, `APITimeoutError`
  2. **서버가 에러로 응답함**: 키 오류, 잘못된 파라미터, 요청 과다 등 → `APIStatusError`와 그 하위 클래스 (HTTP 상태 코드별)
- **SDK가 자동으로 재시도하는 에러가 있다.** 연결 오류, 시간 초과, 408, 409, 429, 5xx는 기본 **2번** 재시도한 뒤에도 실패해야 예외가 발생한다. (`max_retries`로 조절, `0`이면 재시도 안 함)

## 예외 클래스 구조

```
anthropic.APIError
├── APIConnectionError          서버에 닿지 못함
│   └── APITimeoutError         시간 초과
└── APIStatusError              서버가 에러 코드로 응답함
    ├── BadRequestError         400
    ├── AuthenticationError     401
    ├── PermissionDeniedError   403
    ├── NotFoundError           404
    ├── RequestTooLargeError    413
    ├── RateLimitError          429
    ├── OverloadedError         529
    └── InternalServerError     500 이상 (529 제외)
```

## 에러별 정리

| 상태 코드 | 예외 클래스 | 자동 재시도 | 흔한 원인 | 해결 |
|---|---|---|---|---|
| - | `APIConnectionError` | O | 인터넷 끊김, 방화벽, 잘못된 주소 | 네트워크 확인 |
| - | `APITimeoutError` | O | 응답이 제한 시간 안에 안 옴 | timeout 늘리기, 긴 응답은 스트리밍 |
| 400 | `BadRequestError` | X | 파라미터 오류, 빈 메시지, 모델이 지원하지 않는 옵션 | 에러 메시지대로 요청 수정 |
| 401 | `AuthenticationError` | X | API Key 오타, 만료, 삭제 | `.env`의 키 확인 |
| 403 | `PermissionDeniedError` | X | 키에 해당 작업 권한 없음 | Console에서 권한 확인 |
| 404 | `NotFoundError` | X | 모델 ID 오타 (`claude-opus-5.5` 등) | 01의 모델 목록에서 ID 확인 |
| 413 | `RequestTooLargeError` | X | 요청 크기 초과 (큰 이미지, 긴 문서) | 입력 줄이기 |
| 429 | `RateLimitError` | O | 분당 요청/토큰 한도 초과 | 잠시 후 재시도 (`retry-after` 헤더) |
| 500 | `InternalServerError` | O | Anthropic 서버 문제 | 잠시 후 재시도, [status.anthropic.com](https://status.anthropic.com) 확인 |
| 529 | `OverloadedError` | O | 서버 과부하 | 잠시 후 재시도 |

**재시도해도 소용없는 에러(400, 401, 403, 404, 413)는 코드나 설정을 고쳐야 한다.**

In [ ]:
import time

import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()

MODEL = "claude-opus-5-5"


def error_message(e: anthropic.APIStatusError) -> str:
    """에러 응답 본문에서 사람이 읽을 메시지만 꺼냄 (e.message는 본문 전체가 섞여 길다)"""
    if isinstance(e.body, dict):
        return e.body.get("error", {}).get("message", e.message)
    return e.message

## 1. API 연결 확인 (Health Check)

프로그램을 시작할 때 키와 네트워크가 정상인지 먼저 확인하면, 본 작업 중간에 실패하는 일을 줄일 수 있다.
모델 목록 조회(`models.list`)는 **무료**이고 빠르므로 연결 확인용으로 적합하다.

- `limit=1`: 목록을 1개만 받아 응답을 가볍게 함
- `with_options(timeout=10, max_retries=0)`: 확인용이므로 오래 기다리거나 재시도하지 않음

In [ ]:
def check_connection(c: anthropic.Anthropic) -> bool:
    """API 연결 상태를 확인하고 결과를 출력. 정상이면 True"""
    start = time.perf_counter()
    try:
        page = c.with_options(timeout=10, max_retries=0).models.list(limit=1)
    except anthropic.AuthenticationError as e:
        print(f"[실패] 인증 오류 (401): {error_message(e)}")
    except anthropic.PermissionDeniedError as e:
        print(f"[실패] 권한 없음 (403): {error_message(e)}")
    except anthropic.APITimeoutError:
        print("[실패] 시간 초과: 서버 응답이 10초 안에 오지 않음")
    except anthropic.APIConnectionError as e:
        print(f"[실패] 연결 오류: 서버에 닿지 못함 ({e.__cause__})")
    except anthropic.APIStatusError as e:
        print(f"[실패] 서버 에러 ({e.status_code}): {error_message(e)}")
    else:
        elapsed_ms = (time.perf_counter() - start) * 1000
        print(f"[성공] 연결 정상 ({elapsed_ms:.0f} ms, request-id: {page._request_id})")
        return True
    return False


is_connected = check_connection(client)

> **`request-id`란?** 모든 응답(에러 포함)에 붙는 고유 ID다. 문제가 생겨 Anthropic에 문의할 때 이 값을 알려주면 해당 요청을 찾아볼 수 있으므로, 에러 로그에 함께 남겨두면 좋다.

## 2. 인증 오류 (401)

잘못된 키로 클라이언트를 만들어 일부러 실패시킨다.

In [ ]:
bad_client = anthropic.Anthropic(api_key="sk-ant-invalid-key")

try:
    bad_client.models.list(limit=1)
except anthropic.AuthenticationError as e:
    print("예외 클래스 :", type(e).__name__)
    print("상태 코드   :", e.status_code)
    print("메시지      :", error_message(e))
    print("request-id  :", e.request_id)

> 401은 SDK가 재시도하지 않는다. 키가 틀렸다면 몇 번을 다시 보내도 결과가 같기 때문이다.
>
> 01에서 배운 것처럼 `ANTHROPIC_API_KEY`가 아예 없으면 요청 시점에 다른 에러가 난다. 키가 **있지만 틀린** 경우가 401이다.

## 3. Timeout (시간 초과)

`timeout`은 응답을 기다리는 최대 시간(초)이다. **기본값은 10분**으로 넉넉하다.
여기서는 0.01초로 아주 짧게 설정해 일부러 시간 초과를 낸다.

In [ ]:
try:
    client.with_options(timeout=0.01, max_retries=0).models.list(limit=1)
except anthropic.APITimeoutError as e:
    print("예외 클래스 :", type(e).__name__)
    print("메시지      :", e.message)
    print("부모 클래스 :", type(e).__mro__[1].__name__)  # APITimeoutError는 APIConnectionError의 하위 클래스

### timeout 설정 방법

```python
# 클라이언트 전체에 적용
client = anthropic.Anthropic(timeout=60.0)

# 요청 하나에만 적용 (원래 client는 그대로)
client.with_options(timeout=30.0).messages.create(...)

# 단계별로 세밀하게 (연결 2초, 응답 읽기 60초 등)
client = anthropic.Anthropic(timeout=anthropic.Timeout(60.0, connect=2.0))
```

**주의: 재시도까지 포함하면 실제 대기 시간은 더 길어진다.** timeout 30초, `max_retries=2`라면 최악의 경우 30초 × 3번 = 약 90초를 기다린다.

> 응답이 길어서 timeout이 걱정된다면, timeout을 늘리기보다 **스트리밍**을 쓰는 것이 권장된다. 스트리밍은 응답을 조금씩 받기 때문에 연결이 끊길 걱정이 적다.

## 4. 연결 오류 (서버에 닿지 못함)

아무것도 실행되지 않는 주소(`127.0.0.1:9`)로 요청을 보내 연결 실패를 재현한다.
실제 키가 엉뚱한 곳으로 전송되지 않도록 가짜 키를 사용한다.

In [ ]:
offline_client = anthropic.Anthropic(
    api_key="dummy",
    base_url="http://127.0.0.1:9",  # 아무 서버도 없는 주소
    max_retries=0,
)

try:
    offline_client.models.list(limit=1)
except anthropic.APIConnectionError as e:
    print("예외 클래스 :", type(e).__name__)
    print("메시지      :", e.message)
    print("실제 원인   :", repr(e.__cause__))  # 연결 오류의 진짜 원인은 __cause__에 들어 있음

> 연결 오류는 서버 응답이 없으므로 `status_code`와 `request_id`가 **없다.** 실제 원인(DNS 실패, 연결 거부, SSL 오류 등)은 `e.__cause__`로 확인한다.

## 5. 요청 내용 오류: 404, 400

토큰 수만 세는 `count_tokens`(무료)로 재현한다. 모델 ID, 메시지 형식 같은 기본 검사는 메시지 요청과 똑같이 한다.

In [ ]:
# 404: 모델 ID 오타 (점이 아니라 하이픈이 맞음: claude-opus-5-5)
try:
    client.messages.count_tokens(
        model="claude-opus-5.5",
        messages=[{"role": "user", "content": "안녕"}],
    )
except anthropic.NotFoundError as e:
    print(f"[{type(e).__name__} / {e.status_code}] {error_message(e)}")

# 400: 메시지 내용이 비어 있음
try:
    client.messages.count_tokens(
        model=MODEL,
        messages=[{"role": "user", "content": ""}],
    )
except anthropic.BadRequestError as e:
    print(f"[{type(e).__name__} / {e.status_code}] {error_message(e)}")

> 400 에러는 원인이 매우 다양하지만, **에러 메시지가 무엇이 틀렸는지 구체적으로 알려준다.** 메시지를 꼭 출력해서 확인한다.
>
> 단, `count_tokens`가 **모든 규칙을 검사하지는 않는다.** 예를 들어 Opus 5.5에서 400이 나는 `thinking={"type": "disabled"}`도 `count_tokens`에서는 통과한다. `count_tokens`가 통과했다고 실제 요청이 반드시 성공하는 것은 아니다.

## 6. Rate Limit (429)

계정마다 **1분당 요청 수, 1분당 입력/출력 토큰 수** 한도가 있다. 넘으면 `RateLimitError`가 발생한다.

- 반복문으로 요청을 빠르게 많이 보낼 때 주로 발생한다.
- SDK가 자동으로 재시도하므로, 예외가 났다면 **재시도까지 모두 실패한 것**이다.
- 실습 계정에서 일부러 재현하려면 요청을 아주 많이 보내야 하고 요금도 들기 때문에, 여기서는 **현재 한도를 확인하는 방법**과 **처리 코드**만 다룬다.

### 6-1. 현재 한도 확인 (응답 헤더)

한도 정보는 메시지 요청의 **응답 헤더**에 들어 있다. 헤더를 보려면 `.with_raw_response`를 붙여 요청한다.

| 헤더 | 의미 |
|---|---|
| `anthropic-ratelimit-requests-limit` / `-remaining` | 분당 요청 수 한도 / 남은 수 |
| `anthropic-ratelimit-input-tokens-limit` / `-remaining` | 분당 입력 토큰 한도 / 남은 양 |
| `anthropic-ratelimit-output-tokens-limit` / `-remaining` | 분당 출력 토큰 한도 / 남은 양 |
| `...-reset` | 한도가 다시 채워지는 시각 (UTC) |
| `retry-after` | (429일 때) 몇 초 후에 다시 시도하면 되는지 |

> 아래 셀은 짧은 메시지를 1회 보내므로 **소액의 요금이 발생한다.** (무료인 `models.list`, `count_tokens`에는 이 헤더가 오지 않음)

In [ ]:
raw = client.messages.with_raw_response.create(
    model=MODEL,
    max_tokens=16000,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": "안녕"}],
)

# 헤더 중 rate limit 관련 항목만 출력
for key, value in raw.headers.items():
    if key.startswith("anthropic-ratelimit-"):
        print(f"{key:<45} {value}")

# 원래 받던 응답 객체는 .parse()로 꺼냄
response = raw.parse()
print(f"\n응답 출력 토큰: {response.usage.output_tokens}")

### 6-2. RateLimitError 처리 코드

```python
try:
    response = client.messages.create(...)
except anthropic.RateLimitError as e:
    wait = int(e.response.headers.get("retry-after", "60"))  # 서버가 알려준 대기 시간(초)
    print(f"요청 한도 초과. {wait}초 후에 다시 시도하세요.")
```

- SDK가 이미 재시도를 했으므로, 바로 다시 보내기보다 **`retry-after`만큼 기다린 뒤** 보낸다.
- 자동 재시도 횟수를 늘리고 싶다면 `anthropic.Anthropic(max_retries=5)`처럼 설정한다. 직접 재시도 코드를 짤 필요는 거의 없다.
- 한도를 자주 넘는다면 요청 간격을 두거나, 급하지 않은 대량 작업은 **Batch API**(50% 할인, 별도 한도)를 고려한다.

## 7. 종합: 실제로 쓸 수 있는 에러 처리 함수

지금까지의 내용을 하나로 묶는다. 핵심 규칙은 **구체적인 예외부터 먼저 잡는 것**이다.
`APIStatusError`를 먼저 쓰면 그 하위 클래스(401, 404, 429 등)가 전부 거기서 잡혀서 구분할 수 없게 된다.

In [ ]:
def safe_call(request_fn):
    """API 요청 함수를 실행하고, 실패하면 원인을 출력한 뒤 None을 반환"""
    try:
        return request_fn()

    # --- 서버가 에러로 응답한 경우 (구체적인 것부터) ---
    except anthropic.BadRequestError as e:
        print(f"[400] 잘못된 요청 → 요청 내용 수정 필요: {error_message(e)}")
    except anthropic.AuthenticationError as e:
        print(f"[401] 인증 실패 → API Key 확인: {error_message(e)}")
    except anthropic.PermissionDeniedError as e:
        print(f"[403] 권한 없음 → Console 권한 확인: {error_message(e)}")
    except anthropic.NotFoundError as e:
        print(f"[404] 찾을 수 없음 → 모델 ID 확인: {error_message(e)}")
    except anthropic.RateLimitError as e:
        wait = e.response.headers.get("retry-after", "?")
        print(f"[429] 요청 한도 초과 → {wait}초 후 재시도")
    except anthropic.APIStatusError as e:  # 위에서 안 잡힌 나머지 (500, 529 등)
        print(f"[{e.status_code}] 서버 오류 → 잠시 후 재시도 (request-id: {e.request_id})")

    # --- 서버에 닿지 못한 경우 (Timeout이 ConnectionError의 하위라서 먼저) ---
    except anthropic.APITimeoutError:
        print("[Timeout] 응답 시간 초과 → timeout 늘리기 또는 스트리밍 사용")
    except anthropic.APIConnectionError as e:
        print(f"[Connection] 서버에 연결할 수 없음 → 네트워크 확인 ({e.__cause__})")

    return None


# 실패 사례들을 한 번에 확인 (모두 무료 요청)
print("1)", end=" "); safe_call(lambda: bad_client.models.list(limit=1))
print("2)", end=" "); safe_call(lambda: client.messages.count_tokens(model="claude-opus-5.5", messages=[{"role": "user", "content": "안녕"}]))
print("3)", end=" "); safe_call(lambda: client.with_options(timeout=0.01, max_retries=0).models.list(limit=1))
print("4)", end=" "); safe_call(lambda: offline_client.models.list(limit=1))

# 성공 사례
result = safe_call(lambda: client.messages.count_tokens(model=MODEL, messages=[{"role": "user", "content": "안녕"}]))
if result is not None:
    print(f"5) 성공: 입력 토큰 {result.input_tokens}")

> **`lambda`를 쓰는 이유**: `safe_call(client.models.list())`처럼 쓰면 `safe_call`에 들어가기 **전에** 요청이 실행돼서, 에러가 `try` 밖에서 발생한다. `lambda:`로 감싸면 "실행할 방법"만 넘기고, 실제 실행은 `safe_call` 안의 `try`에서 일어난다.

## 정리

| 확인하고 싶은 것 | 방법 |
|---|---|
| 연결이 되는지 | `models.list(limit=1)` (무료) |
| 키가 맞는지 | 위 요청에서 `AuthenticationError` 여부 |
| 요청 기본 형식(모델 ID, 메시지 구조) | `count_tokens` (무료). 모든 규칙을 검사하지는 않음 |
| 남은 Rate Limit | `with_raw_response`로 메시지 요청 후 `anthropic-ratelimit-*` 헤더 |
| 서버 장애 여부 | [status.anthropic.com](https://status.anthropic.com) |